# 00. 角色分工 Agent

多 Agent 系统的基础单元：给每个 Agent 分配一个明确的角色。角色 = 系统提示词 + 可用工具 + 输出契约。

> **检查点**：能写出一个 Agent 角色的完整定义（name、goal、tools、expected_output）。

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'xmake.lua').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'agents'))

from scripts.nb_helpers import setup, chat, chat_json, chat_with_tools, show_trace
client = setup()


🌐 API 模式 — model=deepseek-flash  base_url=https://api.deepseek.com


In [2]:
from dataclasses import dataclass

@dataclass
class Role:
    name: str; goal: str; tools: list[str]
    expected_output: str

@dataclass
class TaskContract:
    task_id: str; description: str
    assigned_to: str; acceptance_criteria: str
    inputs: dict; done: bool = False

# 定义角色
researcher = Role("Researcher","Find accurate facts",["search","scrape"],"Fact list with sources")
writer = Role("Writer","Synthesize into report",["format"],"Markdown report")
reviewer = Role("Reviewer","Verify accuracy",["check"],"Approval or revision notes")

# 任务通过契约传递
contracts = [
    TaskContract("T1","Research climate data","Researcher","At least 3 facts with sources",{"topic":"climate"}),
    TaskContract("T2","Write report","Writer","Markdown with sections",{"facts":"<<T1.output>>"}),
    TaskContract("T3","Review report","Reviewer","All facts verified",{"draft":"<<T2.output>>"}),
]

for c in contracts:
    print(f"{c.task_id} [{c.assigned_to}]: {c.description}")
    print(f"  Criteria: {c.acceptance_criteria}")
    print(f"  Inputs: {c.inputs}")
    c.done = True
    print(f"  Status: {'✓' if c.done else '...'}\n")

T1 [Researcher]: Research climate data
  Criteria: At least 3 facts with sources
  Inputs: {'topic': 'climate'}
  Status: ✓

T2 [Writer]: Write report
  Criteria: Markdown with sections
  Inputs: {'facts': '<<T1.output>>'}
  Status: ✓

T3 [Reviewer]: Review report
  Criteria: All facts verified
  Inputs: {'draft': '<<T2.output>>'}
  Status: ✓



## 角色设计原则

1. **职责单一**：一个角色只做一件事（如 Researcher 只找信息，不写报告）
2. **契约明确**：输入格式、输出格式、完成标准都要写清楚
3. **可替换**：Writer A 和 Writer B 应该能互换，只要满足契约

> **检查点**：能设计 3 个互补的角色，并为它们之间的任务传递设计数据契约。

## 真实 API 实验

同一问题给三个「角色」（不同 system prompt），观察角色设定对输出的真实影响。


In [ ]:
# 角色分工的真实效果
q = "评价这段代码：def f(x): return x/0"
roles = {
    "安全工程师": "你是安全工程师，只关心崩溃、注入、边界情况。",
    "性能工程师": "你是性能工程师，只关心效率与资源消耗。",
    "代码风格审查员": "你是风格审查员，只关心命名、可读性、规范。",
}
for role, system in roles.items():
    out = chat(client, q, system=system, max_tokens=4096)
    print(f"【{role}】{out}\n")


[llm_client] empty content (reasoning consumed budget); retrying with max_output_tokens=4096


## 练习

1. 设计一个「软件开发团队」的 4 个角色：PM、Architect、Developer、Tester。定义每个角色的工具和输出。
2. 实现一个 `handoff(task, from_agent, to_agent)` 函数：将任务及其上下文从一个 Agent 传递给另一个。
3. 思考：如果 Writer 的输出 Reviewer 不满意，信息如何回流？设计反馈循环。

> **检查点**：能为多 Agent 系统设计完整的角色矩阵和任务契约。